[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Indexes and Query Plans


## What you will be able to do

Read the plan SQLite chooses for a query with `EXPLAIN QUERY PLAN`, tell a `SCAN` of every row from
a `SEARCH` through an index, and create the index that turns one into the other. Say in advance
which conditions, joins and sorts an index can serve, from the order of its columns, and which it
cannot, such as a function wrapped around the column. Weigh what an index costs every write, and let
`ANALYZE` and `PRAGMA optimize` give the planner the numbers it chooses by.


## The idea

### The problem

The year of readings is 35,040 rows, and so far every question has read all of them. Finding Oslo's
reading for noon on 1 June means SQLite checking every reading in the table, because nothing tells it
where Oslo's rows are, or where noon is. For one year and four stations that takes a moment. For a
hundred stations over ten years it is 8.76 million rows for every lookup, and a web page that makes a
lookup for every row it shows makes millions of them.

The **Why sqlite3** notebook said that an index lets SQLite find rows without reading the others,
and this notebook measures it. An index only helps the queries that can use it, though, and nothing
in a result says whether one did. A query on `date(hour)` reads every row even when `hour` has an
index. An index on a station and an hour does nothing for a question about an hour at every station.
A join can read a whole table once for every row of the other, and a delete can read the readings to
check a foreign key. A slow query and a fast one return the same rows, so the only way to tell them
apart is to ask SQLite how it means to run them.

### What an index and a query plan are

> An **index** is a structure SQLite keeps beside a table, holding the values of one or more of its
> columns, or of expressions over them, sorted, with the rowid of the row each value came from, so
> that SQLite can find rows by those values without reading the table. SQLite changes every index on
> a table whenever a row of the table changes, and uses an index by itself when a query's conditions
> let it. A **query plan** is SQLite's decision about how to run a statement: which table to read
> first, whether to **`SCAN`** every row or **`SEARCH`** for some of them through an index or by
> rowid, and whether to sort the result with a temporary B-tree. **`EXPLAIN QUERY PLAN`**, written
> in front of a statement, returns the plan instead of running the statement.

### Why it works that way

- **A table is sorted only by its rowid, and an index by its values.** Both are B-trees, which find a
  value in a few steps even among millions. A value in any other column of a table can be found only
  by reading every row.
- **A composite index is sorted by its first column, then by its second.** An index on
  `(station_id, hour)` finds a station, a station and an hour, or a station and a range of hours. An
  hour on its own is spread across every station, so the index cannot find it directly.
- **The index holds the column's values, not what a function makes of them.** A condition on
  `date(hour)` or `station_id + 0` asks about values the index does not hold, so SQLite works them
  out row by row. An index on the expression itself holds exactly those values.
- **An index that holds every column a query needs is covering.** SQLite answers from the index alone
  and never reads the table.
- **Every index costs every write.** An `INSERT`, `UPDATE` or `DELETE` changes each index on the
  table as well as the table, and each index takes space in the file.
- **The planner estimates.** With no statistics, SQLite guesses how many rows an index narrows a
  search to. `ANALYZE` counts them and stores the counts in the table `sqlite_stat1`, a plan can
  change once it has, and `PRAGMA optimize` runs `ANALYZE` when SQLite judges that it would help.
- **A plan is written for people.** SQLite's documentation says the output of `EXPLAIN QUERY PLAN` is
  for interactive debugging, and that its wording can change between releases. It changed in 3.24.0
  and again in 3.36.0, so a plan printed by an older SQLite reads a little differently.

### Where this shows up

Every relational database keeps indexes and explains its plans. PostgreSQL, in the **asyncpg and
psycopg3, Deep Dive** guide, writes `EXPLAIN` in front of a query, and `EXPLAIN ANALYZE` runs it too
and reports what each step cost. The **SQLAlchemy, Deep Dive** guide creates these same indexes from
`index=True` on a column or an `Index` in a model. The **DuckDB, Deep Dive** guide keeps a minimum
and maximum for every block of rows by itself, which does much of an index's work for analytical
queries. An index in a **Pandas, Deep Dive** DataFrame shares the name and little else. In this
guide, every `UNIQUE` constraint in the **Constraints** notebook was enforced by an index SQLite
built for it, and the **Full-Text Search** notebook builds an index of a different kind, for the
words in text.

### What this notebook covers

- `EXPLAIN QUERY PLAN`, and a `SCAN` of every reading
- An index on `(station_id, hour)`, the `SEARCH` it allows, and 200 lookups timed
- What a composite index serves, and what it does not
- Covering indexes
- Joins, and a foreign key checked through an index when a station is deleted
- Sorting, with a temporary B-tree and without one
- Conditions an index cannot help, and an index on an expression
- What indexes cost a load
- `ANALYZE`, `sqlite_stat1` and `PRAGMA optimize`
- When to add an index, and which one
- A slow monthly report, read from its plan and made fast
- Seven errors: an index that already exists, a column misspelled in an index, a unique index over
  duplicates, `'now'` in an index, a composite index in the wrong order, `LIKE` on an indexed column,
  and a plan left over from the statement cache

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id, hour, celsius)")
rows = ((station, f"2025-01-{day:02d}T{hour:02d}:00", 0.0)
        for station in range(1, 5) for day in range(1, 32) for hour in range(24))
conn.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)", rows)

query = "SELECT celsius FROM readings WHERE station_id = 2 AND hour = '2025-01-15T12:00'"
print([step[3] for step in conn.execute("EXPLAIN QUERY PLAN " + query)])

conn.execute("CREATE INDEX readings_by_station_hour ON readings (station_id, hour)")
print([step[3] for step in conn.execute("EXPLAIN QUERY PLAN " + query)])
conn.close()
```

```
['SCAN readings']
['SEARCH readings USING INDEX readings_by_station_hour (station_id=? AND hour=?)']
```

The same query, planned twice. Without an index, SQLite would `SCAN` all 2,976 readings to find one.
With an index on the station and the hour, it would `SEARCH` the index for the one pair asked about,
and `station_id=? AND hour=?` names the columns of the index the search uses. The fourth column of
every row `EXPLAIN QUERY PLAN` returns is the step, in words.


## Setup

Six imports, and the stations and their year of readings, built into the two tables the **Tables and
Queries** notebook designed, with no index on either. A copy of that database, `no_indexes.db`, stays
without indexes, to compare against.

- `sqlite3` builds the database and plans every query
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `time` times lookups, deletes and loads, in the worked examples
- `Path` names the scratch folder and the databases in it
- `shutil` copies the database, and removes the scratch folder at the end

`show_plan` prints the plan for a statement, one step to a line, with a step that belongs to another
indented under it.


In [1]:
import math
import shutil
import sqlite3
import time
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
NO_INDEXES = SCRATCH / "no_indexes.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()
shutil.copy(DATABASE, NO_INDEXES)


def show_plan(conn, sql, parameters=()):
    """Print the plan SQLite chooses for a statement, one step to a line, indented under the step it belongs to."""
    depth = {0: -1}
    for step, parent, _, detail in conn.execute("EXPLAIN QUERY PLAN " + sql, parameters):
        depth[step] = depth[parent] + 1
        print("    " + "  " * depth[step] + detail)


print("built", DATABASE, "and", NO_INDEXES)


built scratch/stations.db and scratch/no_indexes.db


## Worked examples

### A SCAN of every reading

Oslo's reading for noon on 1 June, planned and then fetched. `PRAGMA index_list` lists the indexes on
a table:


In [2]:
conn = sqlite3.connect(DATABASE)
LOOKUP = "SELECT celsius FROM readings WHERE station_id = ? AND hour = ?"
NOON = (ids["Oslo"], "2025-06-01T12:00")

show_plan(conn, LOOKUP, NOON)
print("Oslo at noon on 1 June:", conn.execute(LOOKUP, NOON).fetchone()[0])
print("indexes on readings:", conn.execute("PRAGMA index_list(readings)").fetchall())


    SCAN readings
Oslo at noon on 1 June: 15.0
indexes on readings: []


`SCAN readings` is a full-table scan: SQLite will read every one of the 35,040 rows and keep the one
that matches. With no index on `readings`, there is no other way to find it. The **Tables and
Queries** notebook's `WHERE` clauses all ran this way.

### An index, and the SEARCH it allows

`CREATE INDEX` names the index, the table and the columns, in order. Here 200 lookups of a station
and an hour are timed before the index exists and after, and the cell prints how the two compare,
since the seconds change from machine to machine:


In [3]:
def time_lookups(conn, keys):
    """Seconds taken to look up one reading for every (station_id, hour) in keys."""
    started = time.perf_counter()
    for key in keys:
        conn.execute(LOOKUP, key).fetchone()
    return time.perf_counter() - started


KEYS = [(1 + n % 4, (datetime(2025, 1, 1) + timedelta(hours=n * 977 % 8760)).strftime("%Y-%m-%dT%H:%M"))
        for n in range(200)]
without_index = time_lookups(conn, KEYS)
conn.execute("CREATE INDEX readings_by_station_hour ON readings (station_id, hour)")
with_index = time_lookups(conn, KEYS)

show_plan(conn, LOOKUP, NOON)
print("columns of the index:", conn.execute("PRAGMA index_info(readings_by_station_hour)").fetchall())
print("200 lookups took more than 20 times as long without the index:", without_index > 20 * with_index)


    SEARCH readings USING INDEX readings_by_station_hour (station_id=? AND hour=?)
columns of the index: [(0, 1, 'station_id'), (1, 2, 'hour')]
200 lookups took more than 20 times as long without the index: True


`SEARCH readings USING INDEX readings_by_station_hour (station_id=? AND hour=?)`: SQLite will find
the station and hour in the index, and read only the row it points to. `PRAGMA index_info` lists the
index's columns, in the order they are sorted, with the position each has in the table. On the
machine this notebook was written on, the lookups without the index took more than a hundred times
as long, and the gap grows with the table, since a scan reads every row and a search reads a few
index pages.

### What a composite index serves

An index on `(station_id, hour)` is sorted by station, and within a station by hour. Four questions,
and the plan for each:


In [4]:
for question, sql, parameters in [
    ("one station", "SELECT celsius FROM readings WHERE station_id = ?", (ids["Oslo"],)),
    ("one station in March", "SELECT celsius FROM readings WHERE station_id = ? AND hour >= ? AND hour < ?",
     (ids["Oslo"], "2025-03-01", "2025-04-01")),
    ("two stations at one hour", "SELECT celsius FROM readings WHERE station_id IN (?, ?) AND hour = ?",
     (ids["Oslo"], ids["Bergen"], "2025-06-01T12:00")),
    ("one hour at every station", "SELECT celsius FROM readings WHERE hour = ?", ("2025-06-01T12:00",)),
]:
    print(f"{question}:")
    show_plan(conn, sql, parameters)


one station:
    SEARCH readings USING INDEX readings_by_station_hour (station_id=?)
one station in March:
    SEARCH readings USING INDEX readings_by_station_hour (station_id=? AND hour>? AND hour<?)
two stations at one hour:
    SEARCH readings USING INDEX readings_by_station_hour (station_id=? AND hour=?)
one hour at every station:
    SCAN readings


The index served every question that names the station: one station, a station and a range of hours,
and a list of stations with an hour, which SQLite searches once for each station. It did not serve
the last. An hour on its own sits in four places in the index, one under each station, so SQLite
planned a scan. The first columns of a composite index decide which questions it can answer, which is
why the columns a query always names go first.

### Covering indexes

An index that holds every column a query uses answers the query without reading the table. Three
queries on the same range, asking for different columns:


In [5]:
for columns in ["hour", "COUNT(*)", "celsius"]:
    print(f"SELECT {columns}:")
    show_plan(conn, f"SELECT {columns} FROM readings WHERE station_id = ? AND hour >= ?", (ids["Oslo"], "2025-12-01"))


SELECT hour:
    SEARCH readings USING COVERING INDEX readings_by_station_hour (station_id=? AND hour>?)
SELECT COUNT(*):
    SEARCH readings USING COVERING INDEX readings_by_station_hour (station_id=? AND hour>?)
SELECT celsius:
    SEARCH readings USING INDEX readings_by_station_hour (station_id=? AND hour>?)


`hour` and `COUNT(*)` need nothing the index lacks, so the plan says `COVERING INDEX`, and SQLite
reads only the index. `celsius` is not in the index, so SQLite searches the index and then reads each
row it points to, which is still a search, and slower than a covering one. The column list in the
f-string comes from the code, never from input.

### Joins, and a foreign key checked through an index

A join of stations to their readings, planned in the database with the index and in the copy without
it:


In [6]:
JOIN = """
    SELECT stations.name, readings.hour, readings.celsius
    FROM stations JOIN readings ON readings.station_id = stations.id
    WHERE stations.name = ? AND readings.hour >= ?
"""
bare = sqlite3.connect(NO_INDEXES)
print("with the index:")
show_plan(conn, JOIN, ("Tromso", "2025-12-31"))
print("without it:")
show_plan(bare, JOIN, ("Tromso", "2025-12-31"))


with the index:
    SCAN stations
    SEARCH readings USING INDEX readings_by_station_hour (station_id=? AND hour>?)
without it:
    SCAN readings
    SEARCH stations USING INTEGER PRIMARY KEY (rowid=?)


With the index, SQLite reads the five stations and searches the index for the matching station's
readings. Without it, SQLite reads every reading and looks up each one's station by its
`INTEGER PRIMARY KEY`, which is a search in the other direction. The planner chose the order of the
tables in both cases, and the index decided which order was cheap.

Deleting a station, with foreign keys on, has a hidden read: SQLite has to find out whether any
reading still points at the station, which is why SQLite's documentation recommends an index on the
columns that refer to another table. The plans for a delete, and 50 deletes timed on each database:


In [7]:
def time_deletes(conn):
    """Seconds taken to delete 50 new stations one at a time, each checked against the readings."""
    with conn:
        conn.executemany("INSERT INTO stations (id, name, latitude) VALUES (?, ?, 0.0)",
                         [(1000 + n, f"temporary {n}") for n in range(50)])
    started = time.perf_counter()
    with conn:
        conn.executemany("DELETE FROM stations WHERE id = ?", [(1000 + n,) for n in range(50)])
    return time.perf_counter() - started


for database in (conn, bare):
    database.execute("PRAGMA foreign_keys = ON")
print("with the index:")
show_plan(conn, "DELETE FROM stations WHERE id = ?", (1000,))
print("without it:")
show_plan(bare, "DELETE FROM stations WHERE id = ?", (1000,))
print("50 deletes took more than 10 times as long without the index:", time_deletes(bare) > 10 * time_deletes(conn))
bare.close()


with the index:
    SEARCH stations USING INTEGER PRIMARY KEY (rowid=?)
    SEARCH readings USING COVERING INDEX readings_by_station_hour (station_id=?)
without it:
    SEARCH stations USING INTEGER PRIMARY KEY (rowid=?)
    SCAN readings
50 deletes took more than 10 times as long without the index: True


The plan for a `DELETE` includes the foreign key's check. With the index, SQLite searches it for
readings of the station, reading nothing but the index. Without it, every deleted station means a
scan of all 35,040 readings. `readings_by_station_hour` serves the check because `station_id` is its
first column, so a separate index on `station_id` alone would add a write cost and no speed.

### Sorting, with a temporary B-tree and without one

An `ORDER BY` that an index already follows costs nothing extra. Otherwise SQLite sorts the rows in a
temporary B-tree. Svalbard's readings by hour, then its three coldest readings, before and after an
index on `(station_id, celsius)`:


In [8]:
COLDEST = "SELECT hour, celsius FROM readings WHERE station_id = ? AND celsius IS NOT NULL ORDER BY celsius LIMIT 3"

print("by hour:")
show_plan(conn, "SELECT hour, celsius FROM readings WHERE station_id = ? ORDER BY hour", (ids["Svalbard"],))
print("coldest three:")
show_plan(conn, COLDEST, (ids["Svalbard"],))

conn.execute("CREATE INDEX readings_by_station_celsius ON readings (station_id, celsius)")
print("coldest three, with an index on (station_id, celsius):")
show_plan(conn, COLDEST, (ids["Svalbard"],))
print(conn.execute(COLDEST, (ids["Svalbard"],)).fetchall())


by hour:
    SEARCH readings USING INDEX readings_by_station_hour (station_id=?)
coldest three:
    SEARCH readings USING INDEX readings_by_station_hour (station_id=?)
    USE TEMP B-TREE FOR ORDER BY
coldest three, with an index on (station_id, celsius):
    SEARCH readings USING INDEX readings_by_station_celsius (station_id=? AND celsius>?)
[('2025-01-12T03:00', -17.3), ('2025-01-17T02:00', -17.2), ('2025-01-07T04:00', -17.1)]


Sorting by hour needed no extra step, since the index keeps a station's rows in hour order already.
The coldest three needed `USE TEMP B-TREE FOR ORDER BY`: SQLite would gather all 8,760 of Svalbard's
readings and sort them to keep three. With an index sorted by temperature within a station, the plan
reads the first three entries and stops, and `celsius>?` is SQLite's way of skipping the `NULL`s,
which sort first.

### Conditions an index cannot help

Three conditions that name indexed columns, and still scan:


In [9]:
for question, sql, parameters in [
    ("date() around hour", "SELECT celsius FROM readings WHERE station_id = ? AND date(hour) = ?",
     (ids["Oslo"], "2025-03-01")),
    ("arithmetic on station_id", "SELECT celsius FROM readings WHERE station_id + 0 = ? AND hour = ?",
     (ids["Oslo"], "2025-03-01T12:00")),
    ("OR across two columns", "SELECT celsius FROM readings WHERE station_id = ? OR hour = ?",
     (ids["Oslo"], "2025-03-01T12:00")),
]:
    print(f"{question}:")
    show_plan(conn, sql, parameters)

conn.execute("CREATE INDEX readings_by_day ON readings (date(hour))")
print("date() around hour, with an index on date(hour):")
show_plan(conn, "SELECT station_id, celsius FROM readings WHERE date(hour) = ?", ("2025-03-01",))


date() around hour:
    SEARCH readings USING INDEX readings_by_station_celsius (station_id=?)
arithmetic on station_id:
    SCAN readings
OR across two columns:
    SCAN readings
date() around hour, with an index on date(hour):
    SEARCH readings USING INDEX readings_by_day (<expr>=?)


With `date(hour)`, SQLite still searched an index for the station, then had to work out the date of
every one of Oslo's 8,760 hours, since the index holds hours, not dates. `station_id + 0` holds no
value the index has, so neither column could be used. An `OR` whose second half no index serves
means reading every row anyway. The index on the expression `date(hour)` holds exactly the values
the condition asks about, and the plan shows it as `<expr>=?`. Usually a range,
`hour >= ? AND hour < ?`, is the better fix, since the index that already exists serves it.

### What indexes cost a load

The same year of readings loaded into a new table with no indexes, and into one with three:


In [10]:
ROWS = [(ids[station], hour, celsius) for station, hour, celsius in year_of_readings()]
LOAD = SCRATCH / "load.db"


def timed_load(indexes):
    """Seconds taken to load the year into a new readings table with these indexes, and the file's size."""
    LOAD.unlink(missing_ok=True)
    load = sqlite3.connect(LOAD)
    load.execute("""
        CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL, hour TEXT NOT NULL, celsius REAL)
    """)
    for index in indexes:
        load.execute(index)
    started = time.perf_counter()
    load.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)", ROWS)
    load.commit()
    seconds = time.perf_counter() - started
    load.close()
    return seconds, LOAD.stat().st_size


plain_seconds, plain_size = timed_load([])
indexed_seconds, indexed_size = timed_load([
    "CREATE INDEX by_station_hour ON readings (station_id, hour)",
    "CREATE INDEX by_hour ON readings (hour)",
    "CREATE INDEX by_celsius ON readings (celsius)",
])
print("with three indexes, the load took longer:", indexed_seconds > plain_seconds)
print("and the file is larger:", indexed_size > plain_size)


with three indexes, the load took longer: True
and the file is larger: True


On the machine this notebook was written on, the load with three indexes took almost four times as
long, since every row went into the table and into three sorted indexes. The file grew by the
indexes' pages too. An index is worth its cost when a query that runs often needs it, and a bulk
load into a new table is often faster with the indexes created after it, all at once.

### ANALYZE and PRAGMA optimize

The planner chooses between plans by estimating how many rows each would read. `ANALYZE` counts, and
a plan can change once it has. The question about one hour at every station, before and after:


In [11]:
EVERY_STATION = "SELECT station_id, celsius FROM readings WHERE hour = ?"

print("before ANALYZE:")
show_plan(conn, EVERY_STATION, ("2025-06-01T12:00",))
conn.execute("ANALYZE")
print(conn.execute("SELECT idx, stat FROM sqlite_stat1 WHERE tbl = 'readings' ORDER BY idx").fetchall())
print("after ANALYZE:")
show_plan(conn, EVERY_STATION, ("2025-06-01T12:00",))
print("PRAGMA optimize returned:", conn.execute("PRAGMA optimize").fetchall())


before ANALYZE:
    SCAN readings
[('readings_by_day', '35040 96'), ('readings_by_station_celsius', '35040 8760 35'), ('readings_by_station_hour', '35040 8760 1')]
after ANALYZE:
    SEARCH readings USING INDEX readings_by_station_hour (ANY(station_id) AND hour=?)
PRAGMA optimize returned: []


`sqlite_stat1` holds a row for every index: the number of entries, then the rows a value of the first
column narrows the search to, then the first two columns, and so on. `35040 8760 1` says that a
station narrows the readings to 8,760, and a station with an hour to one. With those numbers SQLite
knows there are only four stations, and plans a skip-scan, `ANY(station_id) AND hour=?`: one search
of the index for each station, in place of a scan of 35,040 rows. `PRAGMA optimize` runs `ANALYZE` on
the tables that it judges need it, and SQLite's documentation recommends running it before closing a
short-lived connection, and after creating indexes.

### When to add an index

| Write | When | Why |
|---|---|---|
| no index | a small table, or a query that reads most of the table anyway | a scan of a few pages costs less than keeping an index up to date |
| an index on the columns a query filters on, most selective first | a query that runs often and keeps few rows | the plan changes from `SCAN` to `SEARCH` |
| a composite index whose first column the queries always name | several queries that share a leading condition | one index serves the column, the pair and a range on the second |
| a covering index | a frequent query that reads only a few columns | SQLite answers from the index and never reads the table |
| an index on the columns that refer to another table | a parent row that is deleted or has its key changed with foreign keys on | the check reads the index, not every child row |
| an index on an expression | a condition on a function of a column that cannot be written as a range | the index holds the computed values the condition asks for |

The default is no index until a plan shows a `SCAN` that matters, and then the one index that turns
it into a `SEARCH`, followed by `ANALYZE` on a small database, or `PRAGMA optimize` on a large one.

### A slow monthly report, made fast

The pieces of this notebook in one job: Tromso's daily coldest, warmest and mean for February, from
a copy of the database without indexes. Read the plan, create the index it needs, and read the plan
again, with the report run 20 times before and after:


In [12]:
REPORT = """
    SELECT date(readings.hour) AS day, MIN(readings.celsius), MAX(readings.celsius), ROUND(AVG(readings.celsius), 1)
    FROM readings JOIN stations ON stations.id = readings.station_id
    WHERE stations.name = ? AND readings.hour >= ? AND readings.hour < ?
    GROUP BY day
    ORDER BY day
"""
FEBRUARY = ("Tromso", "2025-02-01", "2025-03-01")


def time_report(conn):
    """Seconds taken to run the report 20 times, and the rows it returned."""
    started = time.perf_counter()
    for _ in range(20):
        rows = conn.execute(REPORT, FEBRUARY).fetchall()
    return time.perf_counter() - started, rows


report = sqlite3.connect(shutil.copy(NO_INDEXES, SCRATCH / "report.db"))
print("before:")
show_plan(report, REPORT, FEBRUARY)
slow, slow_rows = time_report(report)

report.execute("CREATE INDEX readings_by_station_hour ON readings (station_id, hour)")
report.execute("ANALYZE")
print("after:")
show_plan(report, REPORT, FEBRUARY)
fast, fast_rows = time_report(report)

print("the same rows:", slow_rows == fast_rows, "| first day:", fast_rows[0])
print("20 reports took more than 3 times as long before the index:", slow > 3 * fast)
report.close()


before:
    SCAN readings
    SEARCH stations USING INTEGER PRIMARY KEY (rowid=?)
    USE TEMP B-TREE FOR GROUP BY
after:
    SCAN stations
    SEARCH readings USING INDEX readings_by_station_hour (station_id=? AND hour>? AND hour<?)
    USE TEMP B-TREE FOR GROUP BY
the same rows: True | first day: ('2025-02-01', -8.6, -1.5, -5.3)
20 reports took more than 3 times as long before the index: True


Before the index, the plan read every reading and looked up its station, and then grouped the rows in
a temporary B-tree. After it, SQLite reads the five stations, searches the index for Tromso's
February, 672 readings, and groups only those. The temporary B-tree stays, since the rows are grouped
by `date(hour)`, which the index does not hold, and sorting 672 rows costs little. The rows are the
same, and on the machine this notebook was written on the report ran about six times as fast.

The statistics came from `ANALYZE`, which counts every table. `PRAGMA optimize` analyzes only the
tables whose indexes have no statistics yet, and reads only part of each index, so it would have
left `stations`, which has no index, uncounted. On the SQLite this notebook was written with, those
partial numbers led the planner to a skip-scan through every station's February, which reads four
times the rows. After a new index on a small database, a full `ANALYZE` costs little and counts
everything.

### Where each part came from

| In the report | What it relies on | The section that showed it |
|---|---|---|
| `show_plan(report, REPORT, ...)` | `EXPLAIN QUERY PLAN`, read step by step | A SCAN of every reading |
| `CREATE INDEX ... (station_id, hour)` | a composite index, station first | What a composite index serves |
| `readings.hour >= ? AND readings.hour < ?` | a range the index can search, not a function | Conditions an index cannot help |
| `SEARCH readings USING INDEX` inside the join | the index deciding the order of the tables | Joins, and a foreign key checked through an index |
| `USE TEMP B-TREE FOR GROUP BY` | a sort the index cannot provide | Sorting, with a temporary B-tree and without one |
| `ANALYZE` | statistics for every table, the new index included | ANALYZE and PRAGMA optimize |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/15-indexes-and-query-plans-solutions.ipynb).

**1.** Show the plan for Tromso's readings in January 2025 written with `strftime('%Y-%m', hour)`,
and for the same question written as a range, with the index on `(station_id, hour)` in place. Which
one narrows the search to January?


In [13]:
# your code here


**2.** With the index on `(station_id, hour)` in place, show the plan for
`SELECT MIN(celsius) FROM readings WHERE station_id = ?`, then create the index that lets SQLite
answer it from the index alone, and show the plan again.


In [14]:
# your code here


**3.** Time 100 lookups of one hour at every station, before and after an index on `hour`, and print
whether the lookups took more than 10 times as long before it.


In [15]:
# your code here


**4.** Create an index on the month of every reading, `strftime('%m', hour)`, and show the plan of a
query that counts the readings in December through it.


In [16]:
# your code here


**5.** Print every index in the database, with its table, whether it is unique, and how it was made,
from `PRAGMA index_list`, after creating a table with a `UNIQUE` constraint.


In [17]:
# your code here


**6.** Show whether
`SELECT hour, celsius FROM readings WHERE station_id = ? ORDER BY hour DESC LIMIT 5` needs a
temporary B-tree when the index on `(station_id, hour)` exists.


In [18]:
# your code here


## Common errors

### sqlite3.OperationalError: index readings_by_station_hour already exists


In [19]:
conn.execute("CREATE INDEX readings_by_station_hour ON readings (station_id, hour)")


OperationalError: index readings_by_station_hour already exists

An index's name belongs to the whole database, and a setup script run twice creates it twice. Every
`CREATE` statement has an `IF NOT EXISTS` form, which does nothing when the index is already there:


In [20]:
conn.execute("CREATE INDEX IF NOT EXISTS readings_by_station_hour ON readings (station_id, hour)")
print("indexes on readings:", [row[1] for row in conn.execute("PRAGMA index_list(readings)")])


indexes on readings: ['readings_by_day', 'readings_by_station_celsius', 'readings_by_station_hour']


`IF NOT EXISTS` checks only the name. An index with that name on other columns would stay as it is,
so the name should say what the index holds.

### sqlite3.OperationalError: no such column: stationid


In [21]:
conn.execute("CREATE INDEX readings_by_station ON readings (stationid)")


OperationalError: no such column: stationid

The column is `station_id`. SQLite checks an index's columns against the table when the index is
created, so the misspelling fails straight away, and nothing was created. `PRAGMA table_info` lists
the columns as they are spelled:


In [22]:
print([column[1] for column in conn.execute("PRAGMA table_info(readings)")])
created = conn.execute("SELECT COUNT(*) FROM sqlite_schema WHERE name = 'readings_by_station'").fetchone()[0]
print("readings_by_station exists:", created == 1)


['id', 'station_id', 'hour', 'celsius']
readings_by_station exists: False


No index is needed here anyway: `readings_by_station_hour` starts with `station_id`, so it already
serves every search by station.

### sqlite3.IntegrityError: UNIQUE constraint failed: readings.hour


In [23]:
conn.execute("CREATE UNIQUE INDEX one_reading_an_hour ON readings (hour)")


IntegrityError: UNIQUE constraint failed: readings.hour

A unique index refuses to be created over values that already repeat, and every hour of the year
appears four times, once for each station. The rule meant was one reading for a station and an hour,
and that pair never repeats:


In [24]:
conn.execute("CREATE UNIQUE INDEX one_reading_an_hour ON readings (station_id, hour)")
conn.execute("DROP INDEX readings_by_station_hour")                  # the same columns, now unique
try:
    conn.execute("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)", (*NOON, 21.0))
except sqlite3.IntegrityError as error:
    print("refused:", error)
conn.rollback()


refused: UNIQUE constraint failed: readings.station_id, readings.hour


The unique index serves every search the old index served, so the old one was dropped, to spare
every write its cost. A unique index is how the **Constraints** notebook's `UNIQUE` works, and
creating one on a table that already has rows is how a uniqueness rule is added later, as the
**Changing a Schema** notebook did.

### sqlite3.OperationalError: non-deterministic use of datetime() in an index


In [25]:
conn.execute("CREATE INDEX recent_readings ON readings (hour > datetime('now', '-1 day'))")


OperationalError: non-deterministic use of datetime() in an index

An index stores its values once, when a row is written, and `'now'` is a different moment every time
it is read, so the value would be out of date at once. SQLite accepts `date`, `datetime` and
`strftime` in an index only without `'now'` or `'localtime'`. Work out the moment in Python and
search the plain index with it:


In [26]:
since = (datetime(2025, 12, 31, 23) - timedelta(days=1)).strftime("%Y-%m-%dT%H:%M")
show_plan(conn, "SELECT station_id, celsius FROM readings WHERE station_id = ? AND hour > ?", (ids["Oslo"], since))


    SEARCH readings USING INDEX one_reading_an_hour (station_id=? AND hour>?)


Here the moment is the end of the year, so that the result stays the same; a live program would use
`datetime.now()`.

### No error, and an index never used: its columns in the wrong order


In [27]:
plain = sqlite3.connect(NO_INDEXES)
plain.execute("CREATE INDEX readings_by_hour_station ON readings (hour, station_id)")
OSLO = "SELECT hour, celsius FROM readings WHERE station_id = ?"
show_plan(plain, OSLO, (ids["Oslo"],))


    SCAN readings


The index holds the column the query names, and the plan is still a scan. The index is sorted by hour
first, so Oslo's rows are spread through all of it, one under every hour of the year, and no part of
it holds them together. The column that a query always names has to come first:


In [28]:
plain.execute("DROP INDEX readings_by_hour_station")
plain.execute("CREATE INDEX readings_by_station_hour ON readings (station_id, hour)")
show_plan(plain, OSLO, (ids["Oslo"],))
plain.close()


    SEARCH readings USING INDEX readings_by_station_hour (station_id=?)


### No error, and every station read: LIKE on an indexed column


In [29]:
conn.execute("CREATE INDEX stations_by_name ON stations (name)")
show_plan(conn, "SELECT id, name FROM stations WHERE name LIKE 'Sval%'")


    SCAN stations USING COVERING INDEX stations_by_name


`name` has an index, and a prefix like `'Sval%'` is a range of names, yet the plan scans the whole
index. `LIKE` ignores case, so `'Sval%'` also matches `'SVALBARD'`, while the index is sorted by the
exact characters, in which capital letters come before small ones, so the names that match are not
in one range of it. `GLOB`, which matches case exactly, can use the index, and so can `LIKE` with an
index declared `COLLATE NOCASE`:


In [30]:
print("GLOB:")
show_plan(conn, "SELECT id, name FROM stations WHERE name GLOB 'Sval*'")
conn.execute("CREATE INDEX stations_by_name_nocase ON stations (name COLLATE NOCASE)")
print("LIKE, with an index that ignores case:")
show_plan(conn, "SELECT id, name FROM stations WHERE name LIKE 'Sval%'")


GLOB:
    SEARCH stations USING COVERING INDEX stations_by_name (name>? AND name<?)
LIKE, with an index that ignores case:
    SEARCH stations USING COVERING INDEX stations_by_name_nocase (name>? AND name<?)


### No error, and a plan for an index that is gone: EXPLAIN QUERY PLAN from the statement cache


In [31]:
EVERY_STATION_BY_HOUR = "SELECT station_id FROM readings WHERE hour = ?"
conn.execute("CREATE INDEX readings_by_hour ON readings (hour)")
show_plan(conn, EVERY_STATION_BY_HOUR, ("2025-06-01T12:00",))

conn.execute("DROP INDEX readings_by_hour")
show_plan(conn, EVERY_STATION_BY_HOUR, ("2025-06-01T12:00",))
still_there = conn.execute("SELECT COUNT(*) FROM sqlite_schema WHERE name = 'readings_by_hour'").fetchone()[0]
print("readings_by_hour exists:", still_there == 1)


    SEARCH readings USING INDEX readings_by_hour (hour=?)
    SEARCH readings USING INDEX readings_by_hour (hour=?)
readings_by_hour exists: False


The index was dropped, and the second plan still searched it. sqlite3 keeps the statements it has
prepared, 128 of them by default, and hands back the same one when the same SQL text runs again. A
statement that reads the database is prepared again when the schema has changed, and on the SQLite
this notebook was written with, the cached `EXPLAIN QUERY PLAN` was not. The query itself ran
correctly, since it does read the database. To read a plan after dropping an index, ask through a
new connection, whose cache starts empty:


In [32]:
fresh = sqlite3.connect(DATABASE)
show_plan(fresh, EVERY_STATION_BY_HOUR, ("2025-06-01T12:00",))
fresh.close()
conn.close()


    SCAN readings USING COVERING INDEX one_reading_an_hour


Last, every connection is closed, so this cell removes the scratch folder, with the three databases
in it:


In [33]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `EXPLAIN QUERY PLAN` shows how SQLite will run a statement: `SCAN` reads every row, `SEARCH` reads
  some through an index or the rowid, and `USE TEMP B-TREE` sorts.
- An index on `(station_id, hour)` serves conditions on the station, the station and the hour, and
  the station with a range of hours, but not the hour alone.
- An index that holds every column a query needs is covering, and SQLite never reads the table.
- A function or arithmetic around an indexed column, an `OR` across columns, and a case-insensitive
  `LIKE` on a plain index all scan. Rewrite the condition as a range, or index the expression.
- An index on the columns that refer to another table makes deletes of the parent search, not scan.
- Every index slows every write and takes space, so add the one a plan shows is needed.
- `ANALYZE` stores counts in `sqlite_stat1` that change plans, and `PRAGMA optimize` runs it when it
  judges it useful, before closing a connection and after `CREATE INDEX`.
- A plan read through a cached statement can be stale after `DROP INDEX`, so read it through a new
  connection.


## What is next

The **Full-Text Search** notebook builds a different kind of index, one for the words in text: FTS5,
the `MATCH` operator, and results ranked by how well they match with `bm25`.


---

&#8592; **Previous:** [executemany](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/14-executemany.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Full-Text Search](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/16-full-text-search.ipynb) &#8594;
